In [ ]:
import importlib, json, subprocess, sys

print('=== Inference frameworks pre-installed on Kaggle H100 ===')
INFER_PKGS = [
    'torch', 'transformers', 'accelerate', 'bitsandbytes',
    'auto_gptq', 'autoawq', 'optimum',
    'vllm', 'sglang', 'lmformatenforcer', 'outlines',
    'flash_attn', 'flashinfer', 'xformers', 'triton',
    'safetensors', 'sentencepiece', 'tokenizers', 'huggingface_hub',
    'peft', 'trl', 'datasets',
    'einops', 'numpy', 'pillow',
    'qwen_vl_utils', 'qwen_agent', 'modelscope',
    'ctranslate2', 'lmdeploy', 'tensorrt_llm',
]
results = {}
for name in INFER_PKGS:
    try:
        m = importlib.import_module(name)
        ver = getattr(m, '__version__', None) or getattr(m, 'VERSION', None) or '<no __version__>'
        results[name] = str(ver)
        print(f'  [OK]   {name:<22} {ver}')
    except ImportError:
        results[name] = None
        print(f'  [MISS] {name}')
    except Exception as e:
        results[name] = f'<error:{e!r}>'
        print(f'  [ERR]  {name:<22} {e!r}')

print()
print('=== pip list (relevant subset) ===')
try:
    out = subprocess.check_output([sys.executable, '-m', 'pip', 'list', '--format=json'], timeout=60)
    pkgs = json.loads(out)
    relevant = [p for p in pkgs if any(t in p['name'].lower() for t in (
        'torch','transform','accel','bitsandbytes','vllm','sgl','flash','xform',
        'triton','tokeniz','sentencep','cuda','cudnn','nvidia','peft','gptq',
        'awq','optim','outlines','lmformat','ctranslate','lmdeploy','tensorrt',
        'huggingface','qwen','llama','mistral','gemm'
    ))]
    for p in sorted(relevant, key=lambda x: x['name'].lower()):
        print(f'  {p["name"]:<35} {p["version"]}')
    full_pip = pkgs
except Exception as e:
    full_pip = []
    print(f'pip list failed: {e}')

print()
print('=== H100 Hopper FP8 capability check ===')
try:
    import torch
    p = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
    fp8 = bool(p and p.major >= 9)
    print(f'  device       : {p.name if p else "none"}')
    print(f'  compute      : {p.major}.{p.minor}' if p else '')
    print(f'  fp8_native   : {fp8}')
    print(f'  bf16_dtype   : {torch.bfloat16}')
    print(f'  has float8_e4m3fn : {hasattr(torch, "float8_e4m3fn")}')
    print(f'  has float8_e5m2   : {hasattr(torch, "float8_e5m2")}')
except Exception as e:
    print(f'  torch check failed: {e}')

with open('/kaggle/working/inference_frameworks.json', 'w') as f:
    json.dump({'frameworks': results, 'pip_list': full_pip}, f, indent=2)
print('saved /kaggle/working/inference_frameworks.json')
